# 04 — Heartbeat Segmentation

Using the validated R-peak locations from `03_rpeak_detection.ipynb`, extract a fixed-length window around each peak so we have one array per heartbeat — the unit everything from here on (labeling, features, ML) operates on. No feature extraction yet.

## Load, filter, and detect R peaks

Same pipeline as `03_rpeak_detection.ipynb`, condensed — this notebook picks up from validated R-peak locations rather than re-deriving them step by step.

In [ ]:
import wfdb
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks

RECORD_NAME = "100"


def bandpass_filter(signal, lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    low, high = lowcut / nyquist, highcut / nyquist
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal)


def derivative_filter(signal, fs):
    return np.gradient(signal, 1 / fs)


def moving_window_integration(signal, window_size):
    kernel = np.ones(window_size) / window_size
    return np.convolve(signal, kernel, mode="same")


def detect_r_peaks(filtered_signal, fs, qrs_duration_seconds=0.1,
                    height_frac=0.2, prominence_frac=0.2, max_hr_bpm=200):
    derivative = derivative_filter(filtered_signal, fs)
    squared = derivative ** 2
    window = int(round(qrs_duration_seconds * fs))
    integrated = moving_window_integration(squared, window)

    min_distance = int(round((60 / max_hr_bpm) * fs))
    height = height_frac * np.max(integrated)
    prominence = prominence_frac * np.max(integrated)
    candidates, _ = find_peaks(integrated, distance=min_distance,
                                height=height, prominence=prominence)

    search_radius = window // 2
    refined = []
    for c in candidates:
        start = max(0, c - search_radius)
        end = min(len(filtered_signal), c + search_radius)
        refined.append(start + np.argmax(filtered_signal[start:end]))
    return np.unique(refined)


record = wfdb.rdrecord(RECORD_NAME, pn_dir="mitdb")
fs = record.fs
lead_index = record.sig_name.index("MLII") if "MLII" in record.sig_name else 0
ecg_signal = record.p_signal[:, lead_index]
filtered_signal = bandpass_filter(ecg_signal, 0.5, 40, fs, order=4)

r_peaks = detect_r_peaks(filtered_signal, fs)
print("Number of detected R peaks:", len(r_peaks))

## Converting milliseconds to samples

`n_samples = round((ms / 1000) * fs)`. Array indexing only understands
integer sample positions, so every time-based window has to go through this
conversion before it can be used to slice the signal.

## Choosing the window: 200ms before, 400ms after

- **Before (200ms):** the PR interval (P wave onset to R peak) is typically
  120-200ms. 200ms comfortably captures the full P wave with a small margin.
- **After (400ms):** the QT interval (R peak to T wave end) is typically
  350-440ms. 400ms captures the full T wave with a small margin.
- **Total (600ms) vs. typical RR interval (~800ms at 75bpm):** fits inside
  one heartbeat's worth of time without heavily overlapping neighbors —
  though at unusually fast heart rates some overlap with the next beat's P
  wave is possible. A fixed window is a deliberate simplification; a
  per-beat adaptive window (scaled to each beat's own RR interval) would be
  more precise but is unnecessary complexity for this project.

In [ ]:
pre_ms, post_ms = 200, 400
pre_samples = int(round(pre_ms / 1000 * fs))
post_samples = int(round(post_ms / 1000 * fs))
print(f"pre_samples={pre_samples} ({pre_ms}ms)  post_samples={post_samples} ({post_ms}ms)  "
      f"total_window={pre_samples + post_samples}")

## Extracting beats and handling boundary cases

For an R peak too close to the very start or end of the recording, the
window `[peak - pre_samples : peak + post_samples]` would run outside the
signal array. Rather than zero-pad (which would insert a fake flat segment
that isn't real signal and could quietly distort later features) or wrap
around (which would splice in unrelated signal from the other end of the
recording), the simplest honest choice is to **drop** any beat whose window
doesn't fully fit — we lose a small number of beats at the very edges of a
recording rather than fabricate data for them. On this particular record
none of the detected peaks are close enough to either edge to trigger this,
but the check matters in general (and especially once we run this on
multiple records of varying length).

In [ ]:
beat_waveforms = []
beat_r_peak_samples = []
dropped_count = 0

for p in r_peaks:
    start = p - pre_samples
    end = p + post_samples
    if start < 0 or end > len(filtered_signal):
        dropped_count += 1
        continue
    beat_waveforms.append(filtered_signal[start:end])
    beat_r_peak_samples.append(p)

beat_waveforms = np.array(beat_waveforms)
beat_r_peak_samples = np.array(beat_r_peak_samples)

print("beat_waveforms shape:", beat_waveforms.shape, " (n_beats, window_length)")
print("dropped (boundary) beats:", dropped_count)
print("kept beats:", len(beat_waveforms))

## Storage structure

Two parallel NumPy arrays, aligned by index:

- `beat_waveforms`: shape `(n_beats, window_length)` — the actual segmented
  waveform for each beat. This is exactly the "one row = one heartbeat"
  shape the eventual features DataFrame will need.
- `beat_r_peak_samples`: shape `(n_beats,)` — the original R-peak sample
  index each waveform came from. Keeping this is what lets us trace a beat
  back to its position in the recording — needed next stage to look up the
  nearest annotation and assign a label, and useful any time we want to plot
  a specific beat back in context.

## Plotting a collection of individual beats

Overlaying many beats on a shared time axis **relative to the R peak**
(0 = R peak) is also an implicit sanity check on the segmentation itself:
if the windows were misaligned, the overlay would look smeared rather than
sharply consistent.

In [ ]:
beat_time_axis_ms = (np.arange(pre_samples + post_samples) - pre_samples) / fs * 1000

rng = np.random.default_rng(42)
n_show = 40
indices_to_show = rng.choice(len(beat_waveforms), size=min(n_show, len(beat_waveforms)), replace=False)

fig, ax = plt.subplots(figsize=(10, 6))
for idx in indices_to_show:
    ax.plot(beat_time_axis_ms, beat_waveforms[idx], color="steelblue", alpha=0.3, linewidth=1.0)
ax.axvline(0, color="red", linestyle="--", linewidth=1, label="R peak")
ax.set_title(f"{len(indices_to_show)} individual segmented beats (overlaid)")
ax.set_xlabel("Time relative to R peak (ms)")
ax.set_ylabel("Amplitude (mV)")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig("../results/figures/10_segmented_beats.png", dpi=120)
plt.show()

**What to look for:** a consistent P bump around -150 to -50ms, a tightly
aligned QRS spike right at 0ms, and a consistent T bump roughly 250-350ms
after. Tight alignment across beats (not a smeared/blurred overlay) confirms
the R-peak locations and window are both correct. Next stage: labeling each
of these segmented beats using the MIT-BIH annotations — not feature
extraction yet.